# RAGAS Evaluation: context_precision + context_recall

Loads cached eval_data (from `query_rag_system.ipynb`) and runs context retrieval quality metrics.

**Metrics:**
- `context_precision` — Are the retrieved contexts relevant to the question?
- `context_recall` — Do the contexts cover the ground truth answer?

**Prerequisites:**
- `eval_data_health_wallet.json` generated by `query_rag_system.ipynb`

**See also:** `ragas_eval_answer_metrics.ipynb` for answer_similarity + answer_correctness.

## Setup

In [1]:
import json
import time
from datetime import datetime

import pandas as pd
from llama_stack_client import LlamaStackClient
from rich.pretty import pprint

RAGAS_URL = "http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321"
PROVIDER_ID_INLINE = "trustyai_ragas_inline"

ragas_client = LlamaStackClient(base_url=RAGAS_URL)


def compute_aggregated(score_result):
    """Compute mean from per-question scores, skipping None/NaN entries.
    Falls back to RAGAS aggregated_results if available."""
    agg = score_result.aggregated_results
    if agg is not None and not (isinstance(agg, dict) and None in agg.values()):
        if isinstance(agg, dict):
            vals = [v for v in agg.values() if v is not None]
            return vals[0] if len(vals) == 1 else agg
        return agg
    scores = []
    for row in score_result.score_rows:
        s = row.get("score")
        if s is not None and str(s) != "nan":
            scores.append(float(s))
    return round(sum(scores) / len(scores), 6) if scores else None

In [2]:
# Load eval_data from the main notebook's checkpoint
with open("eval_data_health_wallet.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)
print(f"Loaded {len(eval_data)} evaluation entries")

Loaded 182 evaluation entries


In [3]:
# Find the LLM model ID in RAGAS system
ragas_models = ragas_client.models.list()
ragas_llm_model = None
for m in ragas_models:
    mid = getattr(m, 'identifier', None) or getattr(m, 'id', None)
    mtype = getattr(m, 'model_type', '')
    if hasattr(m, 'custom_metadata') and m.custom_metadata:
        mtype = m.custom_metadata.get('model_type', mtype)
    if mtype == 'llm':
        ragas_llm_model = mid
        break

print(f"LLM model: {ragas_llm_model}")

# Check eval providers
providers = ragas_client.providers.list()
eval_providers = [p for p in providers if p.api == 'eval']
pprint(eval_providers)

INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1/models "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1/providers "HTTP/1.1 200 OK"


LLM model: vllm-inference/Gemma-3-27B-BF16-Distributed


[
│   ProviderInfo(
│   │   api='eval',
│   │   config={
│   │   │   'use_k8s': True,
│   │   │   'base_url': 'https://gemma-3-27b-bf16-distributed-vszp.apps.cluster-5pzpt.5pzpt.sandbox1134.opentlc.com/v1'
│   │   },
│   │   health={'status': 'Not Implemented', 'message': 'Provider does not implement health check'},
│   │   provider_id='trustyai_lmeval',
│   │   provider_type='remote::trustyai_lmeval'
│   ),
│   ProviderInfo(
│   │   api='eval',
│   │   config={
│   │   │   'embedding_model': 'vllm-embedding/qwen3-4b-embedding',
│   │   │   'ragas_config': {'raise_exceptions': False}
│   │   },
│   │   health={'status': 'Not Implemented', 'message': 'Provider does not implement health check'},
│   │   provider_id='trustyai_ragas_inline',
│   │   provider_type='inline::trustyai_ragas'
│   )
]

## Helper: Run a RAGAS Metric

In [4]:
def run_ragas_metric(metric_names, eval_data, label=None):
    """Register dataset + benchmark, run eval, return results or None on failure."""
    label = label or "_".join(metric_names)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    dataset_id = f"hw_{label}_{ts}"
    benchmark_id = f"hw_bench_{label}_{ts}"

    # Register
    ragas_client.beta.datasets.register(
        dataset_id=dataset_id,
        purpose="eval/question-answer",
        source={"type": "rows", "rows": eval_data},
        metadata={"provider_id": "localfs"},
    )
    ragas_client.alpha.benchmarks.register(
        benchmark_id=benchmark_id,
        dataset_id=dataset_id,
        scoring_functions=metric_names,
        provider_id=PROVIDER_ID_INLINE,
    )

    # Run
    job = ragas_client.alpha.eval.run_eval(
        benchmark_id=benchmark_id,
        benchmark_config={
            "eval_candidate": {
                "type": "model",
                "model": ragas_llm_model,
                "sampling_params": {"temperature": 0.1, "max_tokens": 500},
            },
            "scoring_params": {},
        },
    )
    print(f"[{label}] Job {job.job_id} submitted. Metrics: {metric_names}")

    # Poll
    start = time.time()
    while True:
        st = ragas_client.alpha.eval.jobs.status(
            benchmark_id=benchmark_id, job_id=job.job_id
        )
        elapsed = time.time() - start
        print(f"  [{elapsed:.0f}s] {st.status}")
        if st.status in ("completed", "failed"):
            break
        time.sleep(15)

    if st.status == "failed":
        print(f"  FAILED after {elapsed:.0f}s")
        return None

    results = ragas_client.alpha.eval.jobs.retrieve(
        benchmark_id=benchmark_id, job_id=job.job_id
    )
    print(f"  Completed in {elapsed:.0f}s")
    for mn in metric_names:
        if mn in results.scores:
            print(f"  {mn}: {results.scores[mn].aggregated_results}")
    return results

## Run: context_precision + context_recall

In [5]:
results_ctx = run_ragas_metric(
    ["context_precision", "context_recall"],
    eval_data,
    label="context",
)

/tmp/ipykernel_4649/1212626365.py:9: DeprecationWarning: deprecated
  ragas_client.beta.datasets.register(


INFO:httpx:HTTP Request: POST http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1beta/datasets "HTTP/1.1 200 OK"


/tmp/ipykernel_4649/1212626365.py:15: DeprecationWarning: deprecated
  ragas_client.alpha.benchmarks.register(
INFO:httpx:HTTP Request: POST http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_context_20260507_173640/jobs "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_context_20260507_173640/jobs/3 "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_context_20260507_173640/jobs/3 "HTTP/1.1 200 OK"


  [6037s] in_progress


INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_context_20260507_173640/jobs/3 "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_context_20260507_173640/jobs/3/result "HTTP/1.1 200 OK"


  [6057s] completed
  Completed in 6057s
  context_precision: {'context_precision': 0.9682539682249288}
  context_recall: {'context_recall': None}


## Combined Summary (all metrics)

In [6]:
print("=" * 60)
print("ALL METRICS SUMMARY")
print("=" * 60)

print(f"  {'answer_similarity':25s}: (from main notebook)")
print(f"  {'answer_correctness':25s}: (from main notebook)")

if results_ctx:
    for mn in ["context_precision", "context_recall"]:
        if mn in results_ctx.scores:
            sr = results_ctx.scores[mn]
            agg = compute_aggregated(sr)
            scored = sum(1 for r in sr.score_rows if r.get("score") is not None and str(r.get("score")) != "nan")
            skipped = len(sr.score_rows) - scored
            suffix = f" ({skipped} skipped)" if skipped > 0 else ""
            print(f"  {mn:25s}: {agg}{suffix}")
else:
    print(f"  {'context_precision':25s}: FAILED")
    print(f"  {'context_recall':25s}: FAILED")

ALL METRICS SUMMARY
  answer_similarity        : (from main notebook)
  answer_correctness       : (from main notebook)
  context_precision        : 0.9682539682249288
  context_recall           : 0.989266 (5 skipped)


In [7]:
# Per-question detail for successful metrics
all_results = {}
if results_ctx:
    for mn in ["context_precision", "context_recall"]:
        if mn in results_ctx.scores:
            all_results[mn] = results_ctx.scores[mn]

if all_results:
    rows = []
    ref_results = results_ctx
    for i, gen in enumerate(ref_results.generations):
        row = {"question": gen["user_input"][:70]}
        for mn, sr in all_results.items():
            if i < len(sr.score_rows):
                score = sr.score_rows[i].get("score", None)
                if score is not None and str(score) != 'nan':
                    row[mn] = round(score, 3)
                else:
                    row[mn] = "SKIP"
        rows.append(row)

    df = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', 70)
    pd.set_option('display.width', 200)
    print("\nPER-QUESTION SCORES:")
    print(df.to_string(index=False))

    # Count skipped entries
    for mn in all_results:
        if mn in df.columns:
            skips = (df[mn] == "SKIP").sum()
            if skips > 0:
                print(f"\n{mn}: {skips}/{len(df)} entries skipped (parsing failure)")


PER-QUESTION SCORES:
                                                               question  context_precision context_recall
                          Kde nájdem Peňaženku zdravia? Je spoplatnená?              1.000            1.0
               Prečo je Peňaženka zdravia viazaná na mobilnú aplikáciu?              1.000            1.0
 Na čerpanie príspevkov z Peňaženky zdravia musím mať mobilnú aplikáciu              0.917            1.0
 Ak som sa opäť vrátil do VšZP, môžem pre Peňaženku zdravia využívať sv              1.000            1.0
Som váš dlhoročný poistenec, prečo aj ja nemám nárok na Peňaženku\nzdra              0.750            1.0
 Aktualizácia mobilnej aplikácie prebehne automaticky, alebo si novú ve              1.000            1.0
 Máte kontakt na podporu pre klientov? Mám problémy s inštaláciou mobil              1.000            1.0
 Prečo je potrebná aktivácia mobilnej aplikácie? Nie je to zbytočná str              1.000            1.0
    Ak mám nový telefón,

## Save Results

In [8]:
import os

os.makedirs("results", exist_ok=True)

ctx_result = {}
if results_ctx:
    for mn in ["context_precision", "context_recall"]:
        m_data = {"metric": mn, "aggregated": None, "num_scored": 0, "num_skipped": 0, "per_question": []}
        if mn in results_ctx.scores:
            sr = results_ctx.scores[mn]
            m_data["aggregated"] = compute_aggregated(sr)
            for i, gen in enumerate(results_ctx.generations):
                score = sr.score_rows[i].get("score", None) if i < len(sr.score_rows) else None
                is_valid = score is not None and str(score) != "nan"
                if is_valid:
                    m_data["num_scored"] += 1
                else:
                    m_data["num_skipped"] += 1
                m_data["per_question"].append({
                    "question": gen["user_input"],
                    "score": round(float(score), 4) if is_valid else None,
                })
        ctx_result[mn] = m_data

    with open("results/context_metrics.json", "w", encoding="utf-8") as f:
        json.dump(ctx_result, f, ensure_ascii=False, indent=2)
    print("Saved results/context_metrics.json")
    for mn, md in ctx_result.items():
        print(f"  {mn}: mean={md['aggregated']}, scored={md['num_scored']}, skipped={md['num_skipped']}")
else:
    print("Context metrics not available (job failed) — nothing saved")

Saved results/context_metrics.json
  context_precision: mean=0.9682539682249288, scored=182, skipped=0
  context_recall: mean=0.989266, scored=177, skipped=5
